# Cuál es el problema?

Abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere ***predecir cuales clientes van a abandonar el servicio*** (churn) para poder tomar acciones preventivas.

Por qué es importante?

* Conseguir un cliente nuevo representa entre 5x y 25x más que retener uno existente.
* Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.



In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Para transformar columnas específicas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluación para la clasificación

from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt

print("Librerías importadas correctamente")



Librerías importadas correctamente


In [4]:
df = pd.read_csv("abandono_clientes.csv")

In [5]:
print("Primeras 5 filas del dataset")
df.head()

Primeras 5 filas del dataset


,antiguedad_meses,factura_mensual,llamadas_soporte,contrato,satisfaccion,abandono
0,52,61.18,0,Dos_anios,3,0
1,15,54.89,1,Mensual,1,1
2,72,112.95,3,Anual,2,0
3,61,103.06,0,Mensual,1,1
4,21,116.50,2,Mensual,1,1


In [6]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

Filas: 800, Columnas: 6

Tipos de datos:
  antiguedad_meses     -> int64
  factura_mensual      -> float64
  llamadas_soporte     -> int64
  contrato             -> str
  satisfaccion         -> int64
  abandono             -> int64

No hay valores faltantes en el dataset.


In [7]:
df.describe().round(2)

,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion,abandono
count,800.00,800.00,800.00,800.00,800.00
mean,35.53,70.65,2.02,2.97,0.35
std,21.11,29.24,1.42,1.43,0.48
min,1.00,20.46,0.00,1.00,0.00
25%,17.00,43.86,1.00,2.00,0.00
50%,35.00,71.82,2.00,3.00,0.00
75%,54.00,96.10,3.00,4.00,1.00
max,72.00,119.94,7.00,5.00,1.00


In [9]:
#DISTRIBUCION DE LA VARIABLE OBJETIVO
#Si el 99% de los clientes no abandona, es un modelo que tendria una presicion (accuray) completamente inutil
# porque siempre predeciria el 99% de los datos, por tanto hay que balancerar las clases.
# 

conteo=df["abandono"].value_counts()
porcentaje=df['abandono'].value_counts(normalize=True)*100 

print('Distribucion de la variable objetivo(abandono): ')
print(f'Permanece (0) : {conteo[0]} clientes ({porcentaje[0]:.1f}%)')
print(f'Abandona (1) : {conteo[1]} clientes ({porcentaje[1]:.1f}%)')
print()
print('Las clases estan moderadamente desbalanceadas'"regresion-logistica (1).ipynb")
print('Hay mas clientes que permanecen que clientes que abandonan')
print('Esto es lo tipico en la realidad: La mayoria de los clientes no se van')


Distribucion de la variable objetivo(abandono): 
Permanece (0) : 517 clientes (64.6%)
Abandona (1) : 283 clientes (35.4%)

Las clases estan moderadamente desbalanceadasregresion-logistica (1).ipynb
Hay mas clientes que permanecen que clientes que abandonan
Esto es lo tipico en la realidad: La mayoria de los clientes no se van


Analisis exploratorio

In [10]:
comparacion = df.groupby("abandono")[['antiguedad_meses','factura_mensual',
                                      'llamadas_soporte','satisfaccion']].mean().round(2)
comparacion.index = ['Permanece (0)', 'Abandona (1)']

print("Promedio de variables numericas segun abandono ")
comparacion

Promedio de variables numericas segun abandono 


,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion
Permanece (0),39.43,67.02,1.88,3.14
Abandona (1),28.41,77.27,2.28,2.67


In [11]:
#Analisis de tipo de contrato vs abandono

tabla_contrato = pd.crosstab(
    df['contrato'],
    df['abandono'],
    normalize='index'
).round(3) * 100

tabla_contrato.columns = ['Permanece (%)', 'Abandona (%)']
print("Porcentaje de abandono por tipo de contrato")
tabla_contrato


Porcentaje de abandono por tipo de contrato


,Permanece (%),Abandona (%)
contrato,,
Anual,74.0,26.0
Dos_anios,77.3,22.7
Mensual,54.0,46.0


# Preparar los datos para el modelo
1. Separar variables predictoras(x) de la variable objetivo
2.Entrenar los datos con 80% del dataset y probar con el 20% del dataset


In [13]:
#Paso 1: separar x (variables predictorias) e y (variable objetivo)
X = df[['antiguedad_meses', 'factura_mensual', 'llamadas_soporte',
         'satisfaccion','contrato']]
y = df['abandono']

#Paso 2: dividir en conjunto de entrenamiento 80% y prueba 20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, #20% para prueba
    random_state=42, #Semilla para reproducibilidad
    stratify=y
)

print(f'Datos de entrenamiento: {X_train.shape[0]} clientes')
print(f'Datos de prueba: {X_test.shape[0]} clientes')
print('')
print('Proporcion de abandono en entrenamiento: {y_train.mean()*100:.1f}%')
print('Proporcion de abandono en prueba: {y_test.mean()*100:.1f}%')
print('')
print('Las proporciones son similares gracias al parametro stratify=y en train_test_split')


Datos de entrenamiento: 640 clientes
Datos de prueba: 160 clientes

Proporcion de abandono en entrenamiento: {y_train.mean()*100:.1f}%
Proporcion de abandono en prueba: {y_test.mean()*100:.1f}%

Las proporciones son similares gracias al parametro stratify=y en train_test_split


### Construir pipeline y entrenar el modelo
1. Calcula la puntuacion lineal
2. Pasa por esa puntuacion que la convierte en una probabilidad entre 0 y 1
3. Si la probabilidad es mayor a 0,5, predice la clase 1 (abandona); si no, predice 0 (permanece)

In [ ]:
#Definir columnas por tipo

from sklearn.pipeline import Pipeline


numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

#Pipeline: procesamiento+modelo
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

#Entrenar el modelo
pipe.fit(X_train, y_train)

print("Modelo entrenado correctamente")



Modelo entrenado correctamente


### Hacer predicciones 
## Usamos el modelo entrenado para predecir el abandono en los datos de prueba


In [18]:
# Generar predicciones con los datos de prueba
y_pred = pipe.predict(X_test)

#Mostrar las primeras 10 predicciones
comparacion_pred = pd.DataFrame({
    "Real": y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto":  ["Si" if r == p else "NO" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs valores reales:")
print()
comparacion_pred

Primeras 10 predicciones vs valores reales:



,Real,Prediccion,Correcto
0,1,1,Si
1,0,0,Si
2,0,0,Si
3,0,0,Si
4,0,0,Si
5,0,0,Si
6,0,0,Si
7,0,0,Si
8,0,0,Si
9,0,0,Si


# Evaluacion del modelo: Exactitud/Presicion (Acurracy)
Cuando hablamos del porcentaje de presicion nos referimos al % de predicciones que fueron correctas
Accurracy = Predicciones correctas/total predicciones

Ejemplo: si el 95% de clientes NO abandona, un modelo que siempre diga 'No abandona' tendria un 95% de presicion, pero jamas detectaria a un cliente en riesgo. Seria inutil.


In [19]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy (exactitud) {acc:.4f}({acc*100:.1f}%%)")
print()
print(f'Esto significa que el modelo acerto {int(acc*len(y_test))} de {len(y_test)} predicciones')


Accuracy (exactitud) 0.7625(76.2%%)

Esto significa que el modelo acerto 122 de 160 predicciones
